# Futures

In python, 'concurrent.futures' is a module that has 2 main tools under it - threadpoolexecutor and processpoolexecutor. It gets the job done about creating thread pools / process pools and efficiently assigning it tasks...

### What problems do pools solve? 

Thread creation is expensive... So far we've seen that a thread's life cycle usually is : thread is created -> thread starts -> thread does the one task that is handed to it (target function) -> thread dies.

pools don't let threads die.. here once a thread completes a job, it goes back to idle, waiting for ANOTHER task, thus eliminating need to create threads for every task.

More succinctly, pools help us **decouple tasks from threads**.

Also, pools / executors don't create a huge reserve of threads / processes from the get go, ready for use... they still do it lazily, creating threads only when a task comes by, and the pool doesn't have enough threads for it. So expect the usual thread-creation overhead in the beginning, but also see that overhead disappear very soon.

### Why the term 'future'?

A future is a promise of a task result... This is useful because we can set a task in motion, while not blocking immediately for its result... We can get other things done in the meantime, and block for the result further down when we absolutely need it.

NOTE: there's a 1-to-1 relation between task and future. A given 'future' object is concerned with the result of only one task.

```
with ThreadPoolExecutor() as executor:
    future1 = executor.submit(task_one)
    future2 = executor.submit(task_two)  # both running concurrently now

    result1 = future1.result()  # now we collect
    result2 = future2.result()
```

here the 'future' is simply the placeholder... we can use this future object to fetch the result later... Without this future concept, we'd be blocking everytime we submit a task to executor, which prevents us from submitting multiple tasks to it, thereby failing to use executors for the one thing we tend to use them for - parallel / background task execution.

### Thread pool executors, and how to use them

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import threading

def task():
    print("Executing our Task")
    result = 0
    i = 0
    for i in range(10):
        result = result + i
    print("I: {}".format(result))
    print("Task Executed {}".format(threading.current_thread()))

def main():
    executor = ThreadPoolExecutor(max_workers=3)
    task1 = executor.submit(task) # this is how to submit tasks to executor (kind of like assigning the target function to a worker thread, but anonymously)
    task2 = executor.submit(task) # the submit() call returns a future object, which has a set of methods we'll cover in a bit.

    executor.shutdown(wait=True) # blocks until all of the tasks currently undertaken by threads / queued have been finished, it then ends the worker threads / releases resources. (similar in intent to join() method for threads). It basically prevents any more tasks from being submitted to executor.

if __name__ == '__main__':
    main()

# fun-fact: worker threads in an executor are by default daemon! Hence it's pretty imp that you shut the executor down definitively... otherwise program will finish executing (main thread will exit), terminating the daemon threads without completing their work... Had they been non-daemon, program would last a little longer, until all of the threads had finished executing their target function.

### Using Thread Executor as a context manager.

Probably the cleanest way of using them

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def task(n):
    print("Processing {}".format(n))

def main():
    print("Starting ThreadPoolExecutor")

    with ThreadPoolExecutor(max_workers=3) as executor:
        future = executor.submit(task, (2))
        future = executor.submit(task, (3))
        future = executor.submit(task, (4))
    print("All tasks complete")

if __name__ == '__main__':
    main()

### Methods under Future objects

1. **result()**: gives us any returned values from the future object. Note that this call blocks until either timeout is crossed, or result is ready. You can use the `timeout` variable too to specify a timeout, for how long to block.. if timeout is crossed, timeout error will be raised.

    ```python
    futureObj.result(timeout=None) # can be none too.
    ```

2. **add_done_callback()**: to add a callback for when your future object is ready with result. Just add the function you want executed to it.

    ```python
    futureObj.add_done_callback(fn)
    ```

3. **running()**: returns true if your future object is currently running (i.e. getting its task executed). if finished / queued, returns false.

    ``` 
    PENDING → RUNNING → FINISHED
                ↑
            running() = True here only
    ```

    ```python
    futureObj.running()
    ```
4. **cancel()**: tries to cancel a future object. Can be cancelled only if not running / completed.

    ```python
    futureObj.cancel()
    ```

5. **exception()**: fetches any exceptions from the future object.

    ```python
    futureObj.exception()
    ```

5. **done()**: returns true if the future is completed or cancelled / threw an exception.

    ```python
    futureObj.done()
    ```


We'll have code samples demonstrating how to use these further down.

### 1. Cancelling the callable using its future object

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def someTask(n):
    print("Executing Task {}".format(n))
    time.sleep(n)
    print("Task {} Finished Executing".format(n))

def main():
    with ThreadPoolExecutor(max_workers=2) as executor:
        task1 = executor.submit(someTask, (1))
        task2 = executor.submit(someTask, (2))
        task3 = executor.submit(someTask, (3))
        task4 = executor.submit(someTask, (4))

        print(task3.cancel()) # should return true, since it's still in queue when you tried cancelling it.
        
if __name__ == '__main__':
    main()

### 2. Fetching results from future objects

In [ ]:
import time
import random
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed

values = [2,3,4,5,6,7,8]
def multiplyByTwo(n):
    time.sleep(random.randint(1,2))
    return n

def done(n):
    print("Done: {}".format(n))

def main():
    with ThreadPoolExecutor(max_workers=3) as executor:
        # futures = []
        # for v in values:
        #     f = executor.submit(multiplyByTwo, (v))
        #     futures.append(f)

        # for f in futures:
        #     done(f.result())

        results = executor.map(multiplyByTwo, values)
        for result in results:
            done(result)
            
if __name__ == '__main__':
    main()

# you can do it the commented way... or you can use the map() function to make it easier for you to fetch all of the results neatly in a list.

### 3. Collecting future objects

In [ ]:
# you can use as_completed(List[futures]) also for better syntax, but honestly it's just the same...

### 4. Using Callbacks

Here, callbacks refer to functions scheduled to run when a given task is done. you add these callbacks to the future tied to that task.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def task(n):
    print("Processing {}".format(n))

def firstTaskDone(fn):
    if fn.cancelled():
        print("Our {} Future has been cancelled".format(fn.arg))
    elif fn.done():
        print("Our First Task after future has completed")

def secondTaskDone(fn):
    if fn.cancelled():
        print("Our {} Future has been cancelled".format(fn.arg))
    elif fn.done():
        print("Our Second Task after future has completed")

def main():
    print("Starting ThreadPoolExecutor")
    with ThreadPoolExecutor(max_workers=3) as executor:
        future = executor.submit(task, (2))
        future.add_done_callback(firstTaskDone)
        future.add_done_callback(secondTaskDone) # you can also chain callbacks! (order is decided by the order you added the callback to future obj)

    print("All tasks complete")
    
if __name__ == '__main__':
    main()

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def isEven(n):
    print("Checking if {} is even".format(n))
    if type(n) != int:
        raise Exception("Value entered is not an integer")

    if n % 2 == 0:
        print("{} is even".format(n))
        return True
    else:
        print("{} is odd".format(n))
        return False

def main():
    with ThreadPoolExecutor(max_workers=4) as executor:
        task1 = executor.submit(isEven, (2))
        task2 = executor.submit(isEven, (3))
        task3 = executor.submit(isEven, ('t'))

    for future in as_completed([task1, task2, task3]):
        print("Result of Task: {}".format(future.result()))

if __name__ == '__main__':
    main()

# NOTE: propagation of exceptions from worker threads (child threads) to main thread is handled out of the box by ThreadPoolExecutor, which is awesome

### Process Pool Executors

They're pretty much the same as threadpoolexecutors

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import os

def task():
    print("Executing our Task on Process {}".format(os.getpid()))

def main():
    executor = ProcessPoolExecutor(max_workers=3)
    task1 = executor.submit(task)
    task2 = executor.submit(task)

if __name__ == '__main__':
    main()

### Process executor using context manager

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def task(n):
 print("Processing {}".format(n))

def main():
    print("Starting ThreadPoolExecutor")
    with ProcessPoolExecutor(max_workers=3) as executor:
        future = executor.submit(task, (2))
        future = executor.submit(task, (3))
        future = executor.submit(task, (4))
    print("All tasks complete")

if __name__ == '__main__':
    main()

### Exercise: multi-process program to convert images to b/w using pillow

In [ ]:
from PIL import Image
from concurrent.futures import ProcessPoolExecutor
import time


def conv_image(inp_path: str):
    print(f"starting task for {inp_path}")
    image_file = Image.open(inp_path) # open colour image
    image_file = image_file.convert('1') # convert image to black and white
    out_path = inp_path.removeprefix('temp/')
    image_file.save("results/" + out_path)

def main():
    start = time.time()
    print("Starting ProcessPoolExecutor")
    with ProcessPoolExecutor(max_workers=3) as executor:
        for i in range(0,100):
            future = executor.submit(conv_image, ('temp/im1.png'))
            future = executor.submit(conv_image, ('temp/im2.png'))
            future = executor.submit(conv_image, ('temp/im3.png'))
    print("All tasks complete")
    end = time.time()
    print(f"total time taken = {end - start}")

if __name__ == '__main__':
    main()

### When to use thread executors vs pool executors

You know when to use threads, and when to use processes. Same logic applies here. multi-process programs do computations well, and multi-thread programs tend to do better with io-heavy tasks.

In [ ]:
import timeit
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import ProcessPoolExecutor
import math

PRIMES = [
    112272535095293,
    112582705942171,
    112272535095293,
    115280095190773,
    115797848077099,
    1099726899285419
]

def is_prime(n):
    if n % 2 == 0:
        return False

    sqrt_n = int(math.floor(math.sqrt(n)))
    for i in range(3, sqrt_n + 1, 2):
        if n % i == 0:
            return False
    return True

def main():
    t1 = timeit.default_timer()
    with ProcessPoolExecutor(max_workers=4) as executor:
        for number, prime in zip(PRIMES, executor.map(is_prime, PRIMES)):
            print('%d is prime: %s' % (number, prime))
    print("{} Seconds Needed for ProcessPoolExecutor".format(timeit.default_timer() - t1))

    t2 = timeit.default_timer()
    with ThreadPoolExecutor(max_workers=4) as executor:
        for number, prime in zip(PRIMES, executor.map(is_prime, PRIMES)):
            print('%d is prime: %s' % (number, prime))
    print("{} Seconds Needed for ThreadPoolExecutor".format(timeit.default_timer() - t2))

if __name__ == '__main__':
    main()

# processpoolexecutor does noticeably better.